## Probes — liveness, readiness, startup

A container can be *running* but not *working* — deadlocked, mid-GC pause, or stuck on a dead downstream. Its process never exited, so the kubelet thinks all is well while users see timeouts. **Probes** let the kubelet ask "are you actually OK?" Three kinds, three questions:

- **Liveness — "should I restart you?"** Fails repeatedly → the kubelet **kills and restarts** the container (per `restartPolicy`). For "wedged, only a restart helps."
- **Readiness — "should I send you traffic?"** Fails → container marked **not ready**, `Ready` flips `False`, and any Service **stops routing** to it. **Not restarted.** For "alive but warming up / lost my DB / shedding load."
- **Startup — "finished booting yet?"** Gives slow starters a grace period; while it's failing, **liveness and readiness are suspended.** Once it passes once, it's done.

Each probe uses one of three mechanisms:

```yaml
livenessProbe:
  httpGet: { path: /healthz, port: 8080 }   # 2xx/3xx = pass
  initialDelaySeconds: 5
  periodSeconds: 10
  failureThreshold: 3
readinessProbe:
  tcpSocket: { port: 5432 }                 # can we open a TCP connection?
livenessProbe:
  exec: { command: ["sh","-c","test -f /tmp/healthy"] }  # exit 0 = pass
```

The knobs: `initialDelaySeconds` (wait before first probe), `periodSeconds` (how often), `failureThreshold`, `successThreshold` (readiness), `timeoutSeconds`. Defaults are fine for most apps — **the CKA cares which probe type fits which failure mode**, not exact numbers. Rule of thumb: *liveness restarts, readiness gates traffic, startup buys boot time.* On our map this is the **probes** chip in the **Pod** box — how the kubelet decides "restart" vs "stop routing."